In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Window

w = Window.partitionBy("call_id").orderBy("_ingested_at")

bronze_schema = StructType([
    StructField("call_id",    StringType(), True),
    StructField("caller",     StringType(), True),
    StructField("callee",     StringType(), True),
    StructField("start_ts",   StringType(), True),
    StructField("duration_s", StringType(), True),
    StructField("status",     StringType(), True),
])

bronze = (
    spark.read
    .schema(bronze_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/raw/cdrs_dirty.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

silver = (
    bronze
    .withColumn("duration_int",    F.expr("try_cast(duration_s AS INT)"))
    .withColumn("start_ts_parsed", F.expr("try_cast(start_ts AS TIMESTAMP)"))
    .withColumn(
        "_error_type",
        F.when(
            F.col("caller").isNull() |
            (F.trim(F.col("caller")) == "") |
            (F.upper(F.trim(F.col("caller"))) == "NULL"),
            "null_caller",
        )
        .when(
            F.col("start_ts_parsed").isNull(),
            "bad_timestamp",
        )
        .when(
            F.col("duration_int") < 0,
            "negative_duration",
        )
        .when(
            (F.col("status") == "ANSWERED") &
            (F.col("duration_int") == 0),
            "answered_zero",
        )
    )
        .withColumn("_rn", F.row_number().over(w))
    .withColumn(
        "_reject_reason",
        F.coalesce(
            F.col("_error_type"),
            F.when(F.col("_rn") > 1, F.lit("duplicate")),
        ),
    )
)
bronze.write.mode("overwrite").saveAsTable("bronze_cdrs")
display(silver.groupBy("_reject_reason").count().orderBy("_reject_reason"))

dup_ids = silver.groupBy("call_id").count().filter("count > 1").select("call_id")
display(silver.join(dup_ids, "call_id")
              .select("call_id", "_rn", "caller", "duration_s", "_reject_reason")
              .orderBy("call_id", "_rn"))


_reject_reason,count
null,948
answered_zero,14
bad_timestamp,9
duplicate,10
negative_duration,11
null_caller,20


In [0]:
silver = (
    bronze
    .withColumn("duration_int",    F.expr("try_cast(duration_s AS INT)"))
    .withColumn("start_ts_parsed", F.expr("try_cast(start_ts AS TIMESTAMP)"))
    .withColumn(
        "_error_type",
        F.when(
            F.col("caller").isNull() |
            (F.trim(F.col("caller")) == "") |
            (F.upper(F.trim(F.col("caller"))) == "NULL"),
            "null_caller",
        )
        .when(
            F.col("start_ts_parsed").isNull(),
            "bad_timestamp",
        )
        .when(
            F.col("duration_int") < 0,
            "negative_duration",
        )
        .when(
            (F.col("status") == "ANSWERED") &
            (F.col("duration_int") == 0),
            "answered_zero",
        )
    )
        .withColumn("_rn", F.row_number().over(w))
    .withColumn(
        "_reject_reason",
        F.coalesce(
            F.col("_error_type"),
            F.when(F.col("_rn") > 1, F.lit("duplicate")),
        ),
    )
)

In [0]:
silver_clean = (
    silver
    .filter(F.col("_reject_reason").isNull())
    .select(
        "call_id",
        "caller",
        "callee",
        F.col("start_ts_parsed").alias("start_ts"),
        F.col("duration_int").alias("duration_s"),
        "status",
        "_ingested_at",
        "_source_file",
    )
)

In [0]:
bronze_n     = spark.table("bronze_cdrs").count()
silver_n     = spark.table("silver_cdrs").count()
quarantine_n = spark.table("quarantine_cdrs").count()

print(f"bronze={bronze_n}  silver={silver_n}  quarantine={quarantine_n}")

assert bronze_n == silver_n + quarantine_n, (
    f"Row leak: {bronze_n} in, {silver_n + quarantine_n} out"
)
print("Reconciliation OK")

bronze=1012  silver=948  quarantine=64
Reconciliation OK


In [0]:
spark.table("silver_cdrs").printSchema(). start_ts

root
 |-- call_id: string (nullable = true)
 |-- caller: string (nullable = true)
 |-- callee: string (nullable = true)
 |-- start_ts: timestamp (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



---------------------------------------------------------------------------
AttributeError Traceback (most recent call last)
File , line 1
----> 1 spark.table("silver_cdrs").printSchema(). start_ts

AttributeError: 'NoneType' object has no attribute 'start_ts'

In [0]:
quarantine = (
    silver
    .filter(F.col("_reject_reason").isNotNull())
    .select(
        "call_id",
        "caller",
        "callee",
        "start_ts",
        "duration_s",
        "status",
        "_ingested_at",
        "_source_file",
        "_reject_reason",
    )
    .withColumn("_quarantined_at", F.current_timestamp())
)

quarantine.write.mode("overwrite").saveAsTable("quarantine_cdrs")
print(quarantine.count())


### Reconciliation: quarantine vs injected counts

Two injected duplicates copied rows that were already corrupted. Because validation errors take priority over duplicate detection, both copies are labelled with the original error, which moves 2 rows from duplicate into null_caller and negative_duration. Total quarantined rows (64) match total injected issues.

Total quarantined rows (64) match total injected issues (64).

In [0]:
display(spark.table("quarantine_cdrs").filter(F.col("_reject_reason") == "bad_timestamp"))

call_id,caller,callee,start_ts,duration_s,status,_ingested_at,_source_file,_reject_reason,_quarantined_at
2413aad4-8696-4971-a2fa-636ac8c91f87,+48123451012,+48123456848,not_a_timestamp,1326,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
270d5d20-1ced-48d9-812f-abca21996880,+48123451004,+48123456730,not_a_timestamp,778,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
31124205-c646-4826-9261-3d5788ecf09d,+48123451021,+48123456919,not_a_timestamp,797,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
5cc05c1e-459d-4ca2-a140-fe18920a69f9,+48123451020,+48123454952,not_a_timestamp,1338,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
6857288e-573e-4ee5-a438-76f8e3a97650,+48123451019,+48123454606,not_a_timestamp,0,BUSY,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
89f6081d-3db3-4471-8054-9c8b5b656412,+48123451018,+48123454450,not_a_timestamp,522,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
b6e7d95e-a109-4a4d-acdf-3aba5defb782,+48123451024,+48123457765,not_a_timestamp,0,FAILED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
d8ab7fab-72f5-40c8-9700-aea7789da0c0,+48123451014,+48123452133,not_a_timestamp,944,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
fc56058f-872b-4971-a54d-d168a49351c5,+48123451006,+48123457695,not_a_timestamp,871,ANSWERED,2026-10-07T12:24:34.553Z,dbfs:/Volumes/workspace/default/raw/cdrs_dirty.csv,bad_timestamp,2026-10-07T12:24:34.553Z
